# AEGIS — zokastech — Apache 2.0 / MIT

# Entraînement NER PII (XLM-RoBERTa → ONNX)

Ce notebook enchaîne la **génération de données** (synthétique EU), l’**import de jeux complémentaires** (JSONL dans `datasets/training/`), l’**entraînement** Hugging Face, une **publication optionnelle sur le Hugging Face Hub** (`push_hf_model.py`) et l’**export ONNX** — même pipeline que [`training/README.md`](../../training/README.md).

**Prérequis** : Python 3.10+, idéalement GPU (sinon `--cpu` / petits batchs). **Ne pas** coller de vraies PII : utilisez des données synthétiques ou anonymisées.

**Astuce** : exécutez le noyau Jupyter avec le répertoire de travail à la **racine du dépôt** `aegis`, ou laissez la première cellule détecter `REPO_ROOT`. Le dossier du dépôt `datasets/` n’est pas le paquet Hugging Face : les imports passent par `training/ensure_hf_datasets.py` (plus d’erreur `ClassLabel` / `unknown location`).

In [14]:
from pathlib import Path
import os
import sys

def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for d in [p, *p.parents]:
        if (d / "training" / "dataset_builder.py").is_file():
            return d
    raise FileNotFoundError("Impossible de trouver la racine du dépôt (fichier training/dataset_builder.py).")

REPO_ROOT = find_repo_root()
TRAINING_DIR = REPO_ROOT / "training"
DATA_DIR = TRAINING_DIR / "data"
OUTPUT_DIR = TRAINING_DIR / "outputs"
EXPORT_DIR = TRAINING_DIR / "exports" / "onnx_ner_nb"

os.chdir(REPO_ROOT)
sys.path.insert(0, str(TRAINING_DIR))

print("REPO_ROOT =", REPO_ROOT)
print("TRAINING_DIR =", TRAINING_DIR)

REPO_ROOT = /Users/zouhairkasmi/aegis
TRAINING_DIR = /Users/zouhairkasmi/aegis/training


In [15]:
# Décommentez si les dépendances manquent dans l'environnement du noyau
# %pip install -q -r training/requirements.txt

import subprocess
import sys

def pip_install_training():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(TRAINING_DIR / "requirements.txt")])

# pip_install_training()  # décommentez une fois si besoin
print("Python:", sys.executable)

Python: /Users/zouhairkasmi/aegis/.venv/bin/python


## 1. Jeu synthétique (IOB2, multilingue UE)

Paramètres **réduits** pour une démo rapide. Pour une qualité proche production, viser `num_examples >= 50_000` (voir CLI `dataset_builder.py`).

In [26]:
from dataset_builder import build_dataset

NUM_SYNTHETIC = 2_000  # augmentez (ex. 50_000) pour un vrai entraînement
SEED = 42
SYNTH_PATH = DATA_DIR / "notebook_eu_pii_synthetic"

SYNTH_PATH.parent.mkdir(parents=True, exist_ok=True)
ds_synth = build_dataset(NUM_SYNTHETIC, seed=SEED)
ds_synth.save_to_disk(str(SYNTH_PATH))
print(ds_synth)
print("Sauvegardé:", SYNTH_PATH)

Saving the dataset (1/1 shards): 100%|██████████| 100/100 [00:00<00:00, 1528.51 examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 1900
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 100
    })
})
Sauvegardé: /Users/zouhairkasmi/aegis/training/data/notebook_eu_pii_synthetic


## 2. Jeux complémentaires (JSONL) — autres corpus / modèles

Les dossiers [`datasets/training/ner_custom/`](../../datasets/training/ner_custom/) et [`datasets/training/ner_financial_seed/`](../../datasets/training/ner_financial_seed/) contiennent des **échantillons** à étendre. Vous pouvez ajouter vos propres `.jsonl` (même schéma : `tokens` + `ner_tags` IOB2, labels = `training/dataset_builder.py` → `LABELS`).

- **Un modèle spécialisé** : entraînez uniquement sur un `DatasetDict` issu de votre JSONL (gros volume recommandé).
- **Fusion** : concaténez synthétique + custom avec `merge_hf_datasets.py` pour garder la diversité EU tout en renforçant un domaine (finance, santé, etc.).

In [27]:
import subprocess
import sys

CUSTOM_JSONL = [
    REPO_ROOT / "datasets" / "training" / "ner_custom" / "samples.jsonl",
    REPO_ROOT / "datasets" / "training" / "ner_financial_seed" / "samples.jsonl",
]
CUSTOM_HF_PATH = DATA_DIR / "notebook_from_jsonl"
MERGED_PATH = DATA_DIR / "notebook_merged_synth_custom"

subprocess.check_call(
    [
        sys.executable,
        str(TRAINING_DIR / "jsonl_to_hf_dataset.py"),
        *[str(p) for p in CUSTOM_JSONL],
        "--output",
        str(CUSTOM_HF_PATH),
        "--val_ratio",
        "0.2",
        "--seed",
        str(SEED),
    ],
    cwd=str(TRAINING_DIR),
)

subprocess.check_call(
    [
        sys.executable,
        str(TRAINING_DIR / "merge_hf_datasets.py"),
        str(SYNTH_PATH),
        str(CUSTOM_HF_PATH),
        "--output",
        str(MERGED_PATH),
    ],
    cwd=str(TRAINING_DIR),
)

print("Jeu fusionné prêt:", MERGED_PATH)

Saving the dataset (1/1 shards): 100%|██████████| 2/2 [00:00<00:00, 345.77 examples/s]


OK — 10 exemples → /Users/zouhairkasmi/aegis/training/data/notebook_from_jsonl
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 8
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 2
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 102/102 [00:00<00:00, 20884.50 examples/s]


Fusionné 2 jeux → /Users/zouhairkasmi/aegis/training/data/notebook_merged_synth_custom
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 1908
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 102
    })
})
Jeu fusionné prêt: /Users/zouhairkasmi/aegis/training/data/notebook_merged_synth_custom


## 3. Fine-tuning (`train_ner.py`)

Sur **Mac Apple Silicon (MPS)**, un **`MPS backend out of memory`** indique surtout un batch trop grand pour la RAM unifiée — ce n’est **pas** une erreur d’export ONNX. Le notebook utilise un batch réduit et l’accumulation de gradients.

Sinon : `FORCE_CPU = True`, `AEGIS_TRAIN_CPU=1`, ou variables `AEGIS_TRAIN_BATCH` / `AEGIS_TRAIN_GRAD_ACCUM`. **CUDA** : ajoutez `--fp16` si vous éditez la commande ; **CPU** : `--cpu` déjà géré sur Mac sans MPS.

In [24]:
import importlib.util
import os
import platform
import subprocess
import sys

# Dépendances manquantes (souvent si le noyau n’a pas tout `training/requirements.txt`)
for _pkg in ("seqeval", "accelerate"):
    if importlib.util.find_spec(_pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg], cwd=str(REPO_ROOT))

MODEL_OUT = OUTPUT_DIR / "ner-notebook-demo"
MODEL_OUT.mkdir(parents=True, exist_ok=True)

# Mac + MPS : petit batch + grad accum pour limiter la RAM unifiée (OOM → baisser AEGIS_TRAIN_BATCH ou CPU).
FORCE_CPU = False
use_cpu_flag = FORCE_CPU or os.environ.get("AEGIS_TRAIN_CPU", "").lower() in ("1", "true", "yes")
if not use_cpu_flag and platform.system() == "Darwin":
    try:
        import torch

        use_cpu_flag = not torch.backends.mps.is_available()
    except ImportError:
        use_cpu_flag = True

if platform.system() == "Darwin":
    BATCH = int(os.environ.get("AEGIS_TRAIN_BATCH", "1"))
    GRAD_ACCUM = int(os.environ.get("AEGIS_TRAIN_GRAD_ACCUM", "8"))
    MAX_SEQ = int(os.environ.get("AEGIS_MAX_SEQ_LENGTH", "384"))
else:
    BATCH = int(os.environ.get("AEGIS_TRAIN_BATCH", "8"))
    GRAD_ACCUM = int(os.environ.get("AEGIS_TRAIN_GRAD_ACCUM", "1"))
    MAX_SEQ = int(os.environ.get("AEGIS_MAX_SEQ_LENGTH", "512"))

EVAL_BATCH = min(BATCH, 2) if platform.system() == "Darwin" and not use_cpu_flag else BATCH

train_cmd = [
    sys.executable,
    str(TRAINING_DIR / "train_ner.py"),
    "--dataset",
    str(MERGED_PATH),
    "--output_dir",
    str(MODEL_OUT),
    "--num_train_epochs",
    "1",
    "--per_device_train_batch_size",
    str(BATCH),
    "--per_device_eval_batch_size",
    str(EVAL_BATCH),
    "--gradient_accumulation_steps",
    str(GRAD_ACCUM),
    "--max_seq_length",
    str(MAX_SEQ),
    "--gradient_checkpointing",
]
if use_cpu_flag:
    train_cmd.append("--cpu")

print(
    "[train]",
    "CPU (--cpu)" if use_cpu_flag else "MPS ou CUDA",
    "| train_batch =",
    BATCH,
    "| grad_accum =",
    GRAD_ACCUM,
    "| max_seq =",
    MAX_SEQ,
    "| eval_batch =",
    EVAL_BATCH,
)
# Sur machine Linux/Windows + NVIDIA : ne pas mettre --cpu ; ajoutez --fp16 (incompatible avec --cpu dans train_ner.py)
print(" ".join(train_cmd))
subprocess.check_call(train_cmd, cwd=str(TRAINING_DIR))

[train] MPS ou CUDA/auto (pas de --cpu) | batch = 6
/Users/zouhairkasmi/aegis/.venv/bin/python /Users/zouhairkasmi/aegis/training/train_ner.py --dataset /Users/zouhairkasmi/aegis/training/data/notebook_merged_synth_custom --output_dir /Users/zouhairkasmi/aegis/training/outputs/ner-notebook-demo --num_train_epochs 1 --per_device_train_batch_size 6 --per_device_eval_batch_size 6


Traceback (most recent call last):
  File "/Users/zouhairkasmi/aegis/training/train_ner.py", line 19, in <module>
    from seqeval.metrics import f1_score, precision_score, recall_score
  File "/Users/zouhairkasmi/aegis/.venv/lib/python3.12/site-packages/seqeval/metrics/__init__.py", line 1, in <module>
    from seqeval.metrics.sequence_labeling import (accuracy_score,
  File "/Users/zouhairkasmi/aegis/.venv/lib/python3.12/site-packages/seqeval/metrics/sequence_labeling.py", line 14, in <module>
    from seqeval.metrics.v1 import SCORES, _precision_recall_fscore_support
  File "/Users/zouhairkasmi/aegis/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py", line 5, in <module>
    from sklearn.exceptions import UndefinedMetricWarning
  File "/Users/zouhairkasmi/aegis/.venv/lib/python3.12/site-packages/sklearn/__init__.py", line 70, in <module>
    from sklearn.base import clone  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/zouhairkasmi/aegis/.venv/lib/python3.12/s

KeyboardInterrupt: 

## 4. Publication sur le Hugging Face Hub (optionnel)

Même principe que `training/README.md` §3 : pousser `best_hf` vers un dépôt modèle après l’entraînement. Authentification : `huggingface-cli login` ou **`HF_TOKEN`**. Renseignez **`HF_REPO_ID`** dans la cellule suivante ou via **`AEGIS_HF_REPO_ID`**.

In [ ]:
import os
import subprocess
import sys

HF_REPO_ID = ""
if not HF_REPO_ID.strip():
    HF_REPO_ID = os.environ.get("AEGIS_HF_REPO_ID", "").strip()
HF_PRIVATE = os.environ.get("AEGIS_HF_PRIVATE", "").lower() in ("1", "true", "yes")

BEST_HF = MODEL_OUT / "best_hf"
if not HF_REPO_ID:
    print("Aucun repo Hub — étape ignorée (voir section 4).")
elif not BEST_HF.is_dir() or not (BEST_HF / "config.json").is_file():
    print("best_hf introuvable — terminez d’abord l’entraînement (section 3).")
else:
    push_cmd = [
        sys.executable,
        str(TRAINING_DIR / "push_hf_model.py"),
        "--model_dir",
        str(BEST_HF),
        "--repo_id",
        HF_REPO_ID,
    ]
    if HF_PRIVATE:
        push_cmd.append("--private")
    print(" ".join(push_cmd))
    subprocess.check_call(push_cmd, cwd=str(TRAINING_DIR))
    print("Hub :", f"https://huggingface.co/{HF_REPO_ID}")

## 5. Export ONNX (consommation par `aegis-ner`)

Produit `model.onnx`, `model_int8.onnx` et `tokenizer_hf/tokenizer.json` — montez-les dans l’image moteur et renseignez `ner.model_path` dans `aegis-config.yaml`.

In [23]:
import subprocess
import sys

BEST_HF = MODEL_OUT / "best_hf"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

export_cmd = [
    sys.executable,
    str(TRAINING_DIR / "export_onnx.py"),
    "--model_dir",
    str(BEST_HF),
    "--out_dir",
    str(EXPORT_DIR),
]
print(" ".join(export_cmd))
subprocess.check_call(export_cmd, cwd=str(TRAINING_DIR))
print("Export terminé:", EXPORT_DIR)

/Users/zouhairkasmi/aegis/.venv/bin/python /Users/zouhairkasmi/aegis/training/export_onnx.py --model_dir /Users/zouhairkasmi/aegis/training/outputs/ner-notebook-demo/best_hf --out_dir /Users/zouhairkasmi/aegis/training/exports/onnx_ner_nb


model_dir introuvable : /Users/zouhairkasmi/aegis/training/outputs/ner-notebook-demo/best_hf
Indiquez le dossier du checkpoint Hugging Face (ex. …/outputs/ner-…/best_hf) après train_ner.py.


CalledProcessError: Command '['/Users/zouhairkasmi/aegis/.venv/bin/python', '/Users/zouhairkasmi/aegis/training/export_onnx.py', '--model_dir', '/Users/zouhairkasmi/aegis/training/outputs/ner-notebook-demo/best_hf', '--out_dir', '/Users/zouhairkasmi/aegis/training/exports/onnx_ner_nb']' returned non-zero exit status 1.

## Suite

- **Hub** : `training/push_hf_model.py`, section 4 du notebook, ou `AEGIS_HF_REPO_ID`.
- Rapport HTML : `python training/evaluate.py --dataset … --model_dir …/best_hf --out_report …` (voir `training/README.md`).
- **Autre architecture** : `train_ner.py --model_name distilbert-base-multilingual-cased` (vérifiez compatibilité tokenizer / ONNX avec la crate Rust).
- **Nouveau domaine** : dupliquez un dossier sous `datasets/training/` avec vos JSONL, convertissez avec `jsonl_to_hf_dataset.py`, fusionnez ou entraînez seul.